In [1]:
# Import the required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
from pathlib import Path

# Check the available device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.11.0+cu130
Device: cuda


In [2]:

from pathlib import Path
import urllib.request

# Dataset URL
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Download the dataset
data_path = Path("tinyshakespeare.txt")

if not data_path.exists():
    urllib.request.urlretrieve(url, data_path)

# Read the text
text = data_path.read_text(encoding="utf-8")

# Explore the dataset
print("Total characters:", len(text))
print("\nFirst 1000 characters:\n")
print(text[:1000])


Total characters: 1115394

First 1000 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak thi

In [3]:

# Find all unique characters
chars = sorted(set(text))

# Vocabulary size
vocab_size = len(chars)

print("Total characters in text:", len(text))
print("Vocabulary size:", vocab_size)

print("\nUnique characters:")
print("".join(chars))

print("\nFirst 50 characters with their IDs:")
for i, char in enumerate(chars[:50]):
    print(f"{repr(char):10} -> {i}")

Total characters in text: 1115394
Vocabulary size: 65

Unique characters:

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

First 50 characters with their IDs:
'\n'       -> 0
' '        -> 1
'!'        -> 2
'$'        -> 3
'&'        -> 4
"'"        -> 5
','        -> 6
'-'        -> 7
'.'        -> 8
'3'        -> 9
':'        -> 10
';'        -> 11
'?'        -> 12
'A'        -> 13
'B'        -> 14
'C'        -> 15
'D'        -> 16
'E'        -> 17
'F'        -> 18
'G'        -> 19
'H'        -> 20
'I'        -> 21
'J'        -> 22
'K'        -> 23
'L'        -> 24
'M'        -> 25
'N'        -> 26
'O'        -> 27
'P'        -> 28
'Q'        -> 29
'R'        -> 30
'S'        -> 31
'T'        -> 32
'U'        -> 33
'V'        -> 34
'W'        -> 35
'X'        -> 36
'Y'        -> 37
'Z'        -> 38
'a'        -> 39
'b'        -> 40
'c'        -> 41
'd'        -> 42
'e'        -> 43
'f'        -> 44
'g'        -> 45
'h'        -> 46
'i'        -> 47
'j'        -> 48


In [4]:

# Character-to-ID mapping
stoi = {char: i for i, char in enumerate(chars)}

# ID-to-character mapping
itos = {i: char for i, char in enumerate(chars)}

# Encode the text into integers
encoded_text = [stoi[char] for char in text]

# Display a small example
sample_text = text[:50]
sample_encoded = encoded_text[:50]

print("Original text:")
print(repr(sample_text))

print("\nEncoded text:")
print(sample_encoded)

# Decode the IDs back into characters
decoded_text = "".join(itos[i] for i in sample_encoded)

print("\nDecoded text:")
print(repr(decoded_text))

print("\nRound-trip successful:", decoded_text == sample_text)

Original text:
'First Citizen:\nBefore we proceed any further, hear'

Encoded text:
[18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1, 54, 56, 53, 41, 43, 43, 42, 1, 39, 52, 63, 1, 44, 59, 56, 58, 46, 43, 56, 6, 1, 46, 43, 39, 56]

Decoded text:
'First Citizen:\nBefore we proceed any further, hear'

Round-trip successful: True


In [5]:

# Convert the encoded text into a PyTorch tensor
data = torch.tensor(encoded_text, dtype=torch.long)

# Split into training and validation data
split_idx = int(0.9 * len(data))

train_data = data[:split_idx]
val_data = data[split_idx:]

print("Total tokens:", len(data))
print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))

print("\nTraining data shape:", train_data.shape)
print("Validation data shape:", val_data.shape)

Total tokens: 1115394
Training tokens: 1003854
Validation tokens: 111540

Training data shape: torch.Size([1003854])
Validation data shape: torch.Size([111540])


In [6]:
# Sequence length the model can see at once
context_length = 128

# Number of sequences processed together
batch_size = 64

print("Context length:", context_length)
print("Batch size:", batch_size)
print("Tokens per batch:", context_length * batch_size)

Context length: 128
Batch size: 64
Tokens per batch: 8192


In [7]:

def get_batch(split):
    # Select the appropriate dataset
    data = train_data if split == "train" else val_data

    # Random starting positions
    ix = torch.randint(
        len(data) - context_length - 1,
        (batch_size,)
    )

    # Input sequences
    x = torch.stack([
        data[i : i + context_length] for i in ix
    ])

    # Target sequences: shifted one character forward
    y = torch.stack([
        data[i + 1 : i + context_length + 1] for i in ix
    ])

    # Move batches to the selected device
    x = x.to(device)
    y = y.to(device)

    return x, y


# Test the function
X_batch, Y_batch = get_batch("train")

print("X batch shape:", X_batch.shape)
print("Y batch shape:", Y_batch.shape)
print("\nFirst input sequence:")
print(X_batch[0].tolist())

print("\nFirst target sequence:")
print(Y_batch[0].tolist())

X batch shape: torch.Size([64, 128])
Y batch shape: torch.Size([64, 128])

First input sequence:
[41, 53, 51, 43, 1, 40, 59, 63, 11, 0, 14, 59, 63, 1, 50, 39, 42, 57, 6, 1, 53, 56, 1, 43, 50, 57, 43, 1, 63, 53, 59, 56, 1, 50, 39, 57, 57, 43, 57, 1, 41, 56, 63, 10, 1, 15, 53, 51, 43, 1, 40, 59, 63, 8, 0, 0, 15, 50, 53, 61, 52, 10, 0, 21, 44, 1, 21, 1, 61, 43, 56, 43, 1, 52, 53, 58, 1, 47, 52, 1, 50, 53, 60, 43, 1, 61, 47, 58, 46, 1, 25, 53, 54, 57, 39, 6, 1, 58, 46, 53, 59, 1, 57, 46, 53, 59, 50, 42, 57, 58, 1, 58, 39, 49, 43, 0, 52, 53, 1, 51, 53, 52, 43, 63, 1, 53, 44, 1]

First target sequence:
[53, 51, 43, 1, 40, 59, 63, 11, 0, 14, 59, 63, 1, 50, 39, 42, 57, 6, 1, 53, 56, 1, 43, 50, 57, 43, 1, 63, 53, 59, 56, 1, 50, 39, 57, 57, 43, 57, 1, 41, 56, 63, 10, 1, 15, 53, 51, 43, 1, 40, 59, 63, 8, 0, 0, 15, 50, 53, 61, 52, 10, 0, 21, 44, 1, 21, 1, 61, 43, 56, 43, 1, 52, 53, 58, 1, 47, 52, 1, 50, 53, 60, 43, 1, 61, 47, 58, 46, 1, 25, 53, 54, 57, 39, 6, 1, 58, 46, 53, 59, 1, 57, 46, 53, 59, 

In [8]:

# Get the first sequence from the batch
x_sample = X_batch[0].tolist()
y_sample = Y_batch[0].tolist()

# Decode token IDs into characters
x_text = "".join(itos[token_id] for token_id in x_sample)
y_text = "".join(itos[token_id] for token_id in y_sample)

print("Input text (X):")
print(repr(x_text))

print("\nTarget text (Y):")
print(repr(y_text))

print("\nDoes Y start with X shifted by one character?")
print(y_text[:-1] == x_text[1:])

Input text (X):
'come buy;\nBuy lads, or else your lasses cry: Come buy.\n\nClown:\nIf I were not in love with Mopsa, thou shouldst take\nno money of '

Target text (Y):
'ome buy;\nBuy lads, or else your lasses cry: Come buy.\n\nClown:\nIf I were not in love with Mopsa, thou shouldst take\nno money of m'

Does Y start with X shifted by one character?
True


In [9]:

embed_dim = 128

token_embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embed_dim
).to(device)

sample_tokens = X_batch[:1, :5]

sample_embeddings = token_embedding(sample_tokens)

print("Token IDs shape:", sample_tokens.shape)
print("Token embeddings shape:", sample_embeddings.shape)

Token IDs shape: torch.Size([1, 5])
Token embeddings shape: torch.Size([1, 5, 128])


In [12]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, context_length):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size, embed_dim
        )

        self.position_embedding = nn.Embedding(
            context_length, embed_dim
        )

    def forward(self, x):
        token_embeds = self.token_embedding(x)

        positions = torch.arange(
            x.size(1), device=x.device
        )

        position_embeds = self.position_embedding(positions)

        return token_embeds + position_embeds


# Create and test the class
embed_dim = 128

input_embedding = InputEmbedding(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    context_length=context_length
).to(device)

sample_tokens = X_batch[:1, :5]
sample_embeddings = input_embedding(sample_tokens)

print("Input shape:", sample_tokens.shape)
print("Output shape:", sample_embeddings.shape)
print("Position embedding shape:", input_embedding.position_embedding.weight.shape)

Input shape: torch.Size([1, 5])
Output shape: torch.Size([1, 5, 128])
Position embedding shape: torch.Size([128, 128])


In [13]:
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):
        seq_length = x.size(1)

        # Prevent each position from seeing future positions
        causal_mask = torch.triu(
            torch.ones(
                seq_length, seq_length,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        output, _ = self.attention(
            x, x, x,
            attn_mask=causal_mask,
            need_weights=False
        )

        return output


# Create and test the class
num_heads = 4

attention = CausalMultiHeadAttention(
    embed_dim=embed_dim,
    num_heads=num_heads
).to(device)

attention_output = attention(sample_embeddings)

print("Input shape:", sample_embeddings.shape)
print("Attention output shape:", attention_output.shape)
print("Number of heads:", num_heads)
print("Dimension per head:", embed_dim // num_heads)

Input shape: torch.Size([1, 5, 128])
Attention output shape: torch.Size([1, 5, 128])
Number of heads: 4
Dimension per head: 32


In [18]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        self.attention = CausalMultiHeadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # Feed Forward Network inside TransformerBlock
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))

        return x


# Create and test the class
transformer_block = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads
).to(device)

block_output = transformer_block(sample_embeddings)

print("Input shape:", sample_embeddings.shape)
print("Output shape:", block_output.shape)

Input shape: torch.Size([1, 5, 128])
Output shape: torch.Size([1, 5, 128])


In [19]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        context_length,
        num_heads,
        num_blocks=2
    ):
        super().__init__()

        self.input_embedding = InputEmbedding(
            vocab_size,
            embed_dim,
            context_length
        )

        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads)
            for _ in range(num_blocks)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)

        self.output_layer = nn.Linear(
            embed_dim, vocab_size
        )

    def forward(self, x):
        x = self.input_embedding(x)

        for block in self.transformer_blocks:
            x = block(x)

        x = self.final_norm(x)
        logits = self.output_layer(x)

        return logits


# Create and test MiniGPT
model = MiniGPT(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    context_length=context_length,
    num_heads=num_heads,
    num_blocks=2
).to(device)

sample_input = X_batch[:1]
sample_output = model(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", sample_output.shape)
print("Vocabulary size:", vocab_size)
print("Number of parameters:", sum(
    p.numel() for p in model.parameters()
))

Input shape: torch.Size([1, 128])
Output shape: torch.Size([1, 128, 65])
Vocabulary size: 65
Number of parameters: 429889


In [21]:
import math

max_steps = 2000
eval_interval = 200
learning_rate = 3e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}

    for split in ["train", "val"]:
        losses = []

        for _ in range(10):
            X, Y = get_batch(split)
            logits = model(X)

            loss = F.cross_entropy(
                logits.reshape(-1, vocab_size),
                Y.reshape(-1)
            )
            losses.append(loss.item())

        results[split] = sum(losses) / len(losses)

    model.train()
    return results


for step in range(max_steps + 1):
    if step % eval_interval == 0:
        losses = estimate_loss()
        print(
            f"Step {step:4d} | "
            f"Train Loss: {losses['train']:.4f} | "
            f"Val Loss: {losses['val']:.4f}"
        )

    if step == max_steps:
        break

    X, Y = get_batch("train")
    logits = model(X)

    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        Y.reshape(-1)
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(), 1.0
    )

    optimizer.step()

Step    0 | Train Loss: 4.3228 | Val Loss: 4.3114
Step  200 | Train Loss: 2.5409 | Val Loss: 2.5511
Step  400 | Train Loss: 2.4404 | Val Loss: 2.4391
Step  600 | Train Loss: 2.3349 | Val Loss: 2.3612
Step  800 | Train Loss: 2.2421 | Val Loss: 2.2649
Step 1000 | Train Loss: 2.1379 | Val Loss: 2.1656
Step 1200 | Train Loss: 2.0448 | Val Loss: 2.1108
Step 1400 | Train Loss: 1.9656 | Val Loss: 2.0326
Step 1600 | Train Loss: 1.9133 | Val Loss: 1.9923
Step 1800 | Train Loss: 1.8606 | Val Loss: 1.9555
Step 2000 | Train Loss: 1.8184 | Val Loss: 1.9240


In [22]:
@torch.no_grad()
def generate(model, max_new_tokens=500):
    model.eval()

    idx = torch.zeros((1, 1), dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]

        logits = model(idx_cond)
        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1)

        next_char = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_char), dim=1)

    model.train()
    return "".join(itos[i] for i in idx[0].tolist())


print(generate(model))


And read;
Wecome. There in Edwars, thou with chaly;
Whom and and liffet; whes's and plint your sights,
No the will, then whis be ofe,
As the not how's and desent fathes
Lord and to fita be mest too sawn
But in comes?

JULIUS:
Hencirst a here, werids bod mily be it not.

Things, and pAsd broly:
What tsextee so toRicio,
Loldimett of exersed it; bent you, I some as uncaliin
Be the eamioff, and man their or mustrue
with and hath in me priph now
Apperver with theme-sids,
Wat's do haven, her saulife,



In [23]:
# Continue training from step 2000 to step 5000
max_steps = 5000
eval_interval = 200

for step in range(2001, max_steps + 1):

    X, Y = get_batch("train")

    logits = model(X)

    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        Y.reshape(-1)
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(), 1.0
    )

    optimizer.step()

    if step % eval_interval == 0:
        losses = estimate_loss()

        print(
            f"Step {step:4d} | "
            f"Train Loss: {losses['train']:.4f} | "
            f"Val Loss: {losses['val']:.4f}"
        )

Step 2200 | Train Loss: 1.7734 | Val Loss: 1.8893
Step 2400 | Train Loss: 1.7347 | Val Loss: 1.8735
Step 2600 | Train Loss: 1.7009 | Val Loss: 1.8632
Step 2800 | Train Loss: 1.6608 | Val Loss: 1.8227
Step 3000 | Train Loss: 1.6586 | Val Loss: 1.8232
Step 3200 | Train Loss: 1.6288 | Val Loss: 1.7871
Step 3400 | Train Loss: 1.6031 | Val Loss: 1.7894
Step 3600 | Train Loss: 1.5911 | Val Loss: 1.7679
Step 3800 | Train Loss: 1.5975 | Val Loss: 1.7557
Step 4000 | Train Loss: 1.5692 | Val Loss: 1.7493
Step 4200 | Train Loss: 1.5699 | Val Loss: 1.7515
Step 4400 | Train Loss: 1.5511 | Val Loss: 1.7385
Step 4600 | Train Loss: 1.5393 | Val Loss: 1.7155
Step 4800 | Train Loss: 1.5350 | Val Loss: 1.7226
Step 5000 | Train Loss: 1.5321 | Val Loss: 1.7161


In [24]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=300):
    model.eval()

    # Encode the prompt into character IDs
    input_ids = [stoi[char] for char in prompt]
    idx = torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=device
    )

    # Generate one character at a time
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]

        logits = model(idx_cond)
        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1)
        next_char = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_char), dim=1)

    model.train()

    return "".join(itos[i] for i in idx[0].tolist())

In [25]:
prompt = "ROMEO:"

generated_text = generate(
    model,
    prompt=prompt,
    max_new_tokens=300
)

print(generated_text)

ROMEO:
As it poword, hear not me?

KING EDWBRANLETH:
This of pent; and not thou on this for sleep.
Hath as 'twen the better, thee will of speak.

KING RICHARD II:
Here welcome, and smould reat this honoure to-mer
By with a did me what your to to que: ha, prince of they
dost. Lord may see his be comemble a
